# PINN 4 — Inverse problem: identifying a diffusion coefficient

In the forward problems the PDE was fully known. Here the diffusion coefficient $D$ in

$$u_t = D\,u_{xx},\qquad x,t\in[0,1],\qquad u(x,0)=\sin(\pi x),\quad u(0,t)=u(1,t)=0$$

is **unknown** and must be recovered from 200 noisy measurements (2 % Gaussian noise),
with exact solution $u=\sin(\pi x)\,e^{-D\pi^2 t}$ and true value $D=0.1$.

$D$ is a trainable `nn.Parameter` of the problem, parameterised as $D=e^{\log D}$ so it
always stays positive. The optimiser updates it together with the network weights.

| | |
|---|---|
| Problem class | `neural_pde_solvers.pinn.DiffusionInverse1D` |
| Config | `configs/pinn/diffusion_inverse1d.yaml` |
| CLI | `pinn-run configs/pinn/diffusion_inverse1d.yaml` |

In [ ]:
# --- Setup -------------------------------------------------------------------
# Locally: `pip install -e ".[pinn]"` from the repository root.
# On Google Colab the package is installed from GitHub automatically.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("neural_pde_solvers") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "neural-pde-solvers[pinn] @ git+https://github.com/tvh0ng/neural-pde-solvers.git",
        ]
    )

import matplotlib.pyplot as plt
import numpy as np
import torch

from neural_pde_solvers import set_seed, setup_logging
from neural_pde_solvers.pinn import (
    MLP,
    PINNTrainer,
    TrainerConfig,
    count_parameters,
    evaluate,
    grid,
    predict,
    resolve_device,
)
from neural_pde_solvers.pinn.plotting import plot_loss_history

setup_logging()
set_seed(42)
device = resolve_device("auto")
QUICK = os.environ.get("PINN_QUICK") == "1"  # shortened run used by CI
print(f"PyTorch {torch.__version__} | device: {device}")

## 1. Synthetic measurements

In [ ]:
from neural_pde_solvers.pinn import DiffusionInverse1D

problem = DiffusionInverse1D(D_true=0.1, D_init=0.5, n_data=200, noise=0.02).to(device)
Xd, ud = problem.X_data.cpu().numpy(), problem.u_data.cpu().numpy()[:, 0]

fig, ax = plt.subplots(figsize=(6, 4.5))
sc = ax.scatter(Xd[:, 0], Xd[:, 1], c=ud, cmap="RdBu_r", s=18)
plt.colorbar(sc, label="measured u")
ax.set(xlabel="x", ylabel="t", title=f"{len(ud)} noisy measurements")
plt.show()
print(f"initial guess D = {problem.physical_parameters()['D']:.3f} (true {problem.D_true})")

## 2. Loss

$$\mathcal L = \text{pde} + 10\,\text{data} + 10\,\text{ic} + 10\,\text{bc}.$$

The **data** term is what makes $D$ identifiable: without measurements any $D$ admits a
solution satisfying the PDE, IC and BC. Conversely the PDE term regularises the fit,
which is why noisy data still yield an accurate $D$.

In [ ]:
model = MLP(in_dim=2, width=64, depth=4, activation="tanh").to(device)
config = TrainerConfig(
    epochs=1000 if QUICK else 15000,
    lr=1e-3,
    n_collocation=5000,
    scheduler="step",
    scheduler_kwargs=dict(step_size=3000, gamma=0.5),
    log_every=1500,
)
history = PINNTrainer(model, problem, config).fit()
D_hat = problem.physical_parameters()["D"]
print(
    f"D estimate {D_hat:.5f} | true {problem.D_true} | "
    f"error {abs(D_hat - problem.D_true) / problem.D_true:.2%}"
)

## 3. Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
axes[0].plot(history["epoch"], history["D"], lw=2, label="estimate")
axes[0].axhline(problem.D_true, color="tab:red", ls="--", label="true D")
axes[0].axhline(problem.D_init, color="gray", ls=":", label="initial guess")
axes[0].set(xlabel="Iteration", ylabel="D", title="Parameter convergence")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
plot_loss_history(history, ax=axes[1])
fig.tight_layout()
plt.show()

In [ ]:
from neural_pde_solvers.pinn.plotting import plot_prediction_vs_exact_2d

mesh, X = grid(problem, 101)
u_pred, u_exact = predict(model, X), problem.exact(X)
print(evaluate(model, problem, 101))
fig, axes = plot_prediction_vs_exact_2d(mesh, u_pred, u_exact, labels=("x", "t"), cmap="RdBu_r")
axes[0].scatter(Xd[:, 0], Xd[:, 1], s=4, c="k", alpha=0.4)
plt.show()

## Key takeaways

1. Unknown physical parameters are just extra trainable variables.
2. Re-parameterise to respect physical constraints ($D = e^{\log D} > 0$).
3. Data make the inverse problem well posed; the PDE regularises noisy data.
4. The network and the parameter are learned **simultaneously**, in one training run.

**Applications:** thermal conductivity and elastic moduli (materials), viscosity and
permeability (fluids), diffusion and reaction rates (biology), model calibration from
sparse observations (climate).